# Honeywell Temperature Alarm Prediction Model
### Time Series Forecasting for Tag: `03TIC_1023.PV` (PVHI Alarm, Threshold = 21.0)

This notebook implements the machine learning pipeline to predict temperature alarms at multiple future horizons: **15 minutes**, **30 minutes**, and **60 minutes**.

#### Key Steps in this Pipeline:
1. **Load and Align Data**: Load the 4-year parquet dataset. Set the timestamp as the index and reindex to a complete 1-minute grid to ensure chronological integrity during feature calculations.
2. **Feature Engineering**:
   - Lags (1m, 2m, 5m, 10m, 15m, 30m, 60m)
   - Rolling statistics (mean, std, min, max over 10m, 30m, 60m windows)
   - Difference / rate of change features (5m, 15m, 30m differences)
   - Time features (hour of day, month, day of week)
3. **Chronological Splitting**: Train (2022-2024), Validation (H1 2025), and Test (H2 2025 - Jan 2026).
4. **LightGBM Regressor Training**: Train gradient-boosted regressors for the four horizons using early stopping on the validation set.
5. **Alarm Classification Evaluation**: Threshold predictions at `21.0` and evaluate Precision, Recall, F1-Score, and False Alarm Rate.

--- 
## 1. Imports and Configuration

In [1]:
import os
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, precision_recall_fscore_support, confusion_matrix
import joblib
import time
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting styles
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
sns.set_style("whitegrid")

# Paths
DATA_PATH = r"03TIC_1023_PVHI/03TIC_1023_PVHI/03TIC_1023_Final_merged_TripDataRemoved.parquet"
MODEL_DIR = r"models"
OUTPUT_DIR = r"outputs"

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Directories set up successfully.")

Directories set up successfully.


--- 
## 2. Data Loading & Time-Alignment
Because trips/shutdowns were removed, there are gaps in the raw data. To avoid computing rolling windows across artificial jumps in time, we reindex the dataset to a complete 1-minute grid.

In [2]:
print("Loading parquet data...")
df = pd.read_parquet(DATA_PATH)
df['TimeStamp'] = pd.to_datetime(df['TimeStamp'])
df = df.sort_values('TimeStamp')

print(f"Raw dataset shape: {df.shape}")
print(f"Timestamp range: {df['TimeStamp'].min()} to {df['TimeStamp'].max()}")

# Set TimeStamp as index and reindex to complete 1-minute frequency
print("Reindexing to regular 1-minute grid...")
df = df.set_index('TimeStamp')
full_idx = pd.date_range(start=df.index.min(), end=df.index.max(), freq='1min')
df = df.reindex(full_idx)
df.index.name = 'TimeStamp'
print(f"Reindexed dataset shape: {df.shape} (Added {len(df) - len(full_idx)} rows as NaNs during gaps)")

# Impute very small gaps (e.g. up to 5 minutes) using forward fill
print("Imputing small gaps (up to 5 mins)...")
df = df.ffill(limit=5)
print("Null counts in target column (03TIC_1023.PV):", df['03TIC_1023.PV'].isnull().sum())

Loading parquet data...
Raw dataset shape: (2019221, 20)
Timestamp range: 2022-01-03 22:45:00 to 2026-01-09 23:29:00
Reindexing to regular 1-minute grid...
Reindexed dataset shape: (2112525, 19) (Added 0 rows as NaNs during gaps)
Imputing small gaps (up to 5 mins)...
Null counts in target column (03TIC_1023.PV): 93011


--- 
## 3. Feature Engineering
We create historical lagged values, rolling statistics, rate-of-change, and time features. We focus on the most highly correlated columns to avoid feature explosion.

In [3]:
target_col = '03TIC_1023.PV'
key_cols = ['03TIC_1023.PV', '03TI_1024.PV', '03TI_1015.PV', '03PIC_1023.PV', '03TI_1081.PV']

# Time features
df['hour'] = df.index.hour
df['month'] = df.index.month
df['dayofweek'] = df.index.dayofweek

# Lag features
print("Creating lag features...")
for col in key_cols:
    for lag in [1, 2, 5, 10, 15, 30, 60]:
        df[f'{col}_lag_{lag}'] = df[col].shift(lag)
        
# Rolling features
print("Creating rolling window features...")
for col in key_cols:
    for window in [10, 30, 60]:
        df[f'{col}_roll_mean_{window}'] = df[col].rolling(window=window, min_periods=int(window*0.8)).mean()
        df[f'{col}_roll_std_{window}'] = df[col].rolling(window=window, min_periods=int(window*0.8)).std()
        df[f'{col}_roll_max_{window}'] = df[col].rolling(window=window, min_periods=int(window*0.8)).max()
        df[f'{col}_roll_min_{window}'] = df[col].rolling(window=window, min_periods=int(window*0.8)).min()

# Rate of change features
print("Creating rate-of-change features...")
for col in key_cols:
    for diff in [5, 15, 30]:
        df[f'{col}_diff_{diff}'] = df[col] - df[col].shift(diff)
        
# Future targets for our 4 horizons
print("Creating future target variables...")
df['target_5m'] = df[target_col].shift(-5)
    df['target_15m'] = df[target_col].shift(-15)
df['target_30m'] = df[target_col].shift(-30)
df['target_60m'] = df[target_col].shift(-60)

print("Feature engineering complete.")

Creating lag features...
Creating rolling window features...
Creating rate-of-change features...


C:\Users\Ajay_ML\AppData\Local\Temp\ipykernel_21140\2718610380.py:28: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{col}_diff_{diff}'] = df[col] - df[col].shift(diff)
C:\Users\Ajay_ML\AppData\Local\Temp\ipykernel_21140\2718610380.py:28: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{col}_diff_{diff}'] = df[col] - df[col].shift(diff)
C:\Users\Ajay_ML\AppData\Local\Temp\ipykernel_21140\2718610380.py:28: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, 

Creating future target variables...
Feature engineering complete.


C:\Users\Ajay_ML\AppData\Local\Temp\ipykernel_21140\2718610380.py:28: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{col}_diff_{diff}'] = df[col] - df[col].shift(diff)
C:\Users\Ajay_ML\AppData\Local\Temp\ipykernel_21140\2718610380.py:28: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{col}_diff_{diff}'] = df[col] - df[col].shift(diff)
C:\Users\Ajay_ML\AppData\Local\Temp\ipykernel_21140\2718610380.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, 

In [4]:
df.head()

,02FI_1000.PV,03FIC_1085.PV,03LIC_1016.PV,03LIC_1034.PV,03PDI_1020.PV,03PDI_1077.PV,03PIC_1013.PV,03PIC_1023.PV,03PIC_1068.PV,03PI_1409.PV,...,03TI_1015.PV_diff_30,03PIC_1023.PV_diff_5,03PIC_1023.PV_diff_15,03PIC_1023.PV_diff_30,03TI_1081.PV_diff_5,03TI_1081.PV_diff_15,03TI_1081.PV_diff_30,target_15m,target_30m,target_60m
TimeStamp,,,,,,,,,,,,,,,,,,,,,
2022-01-03 22:45:00,6.645525,216.82233,36.621506,45.743240,160.16232,0.221155,95.075195,6.606513,24.117039,25.977545,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17.952660,17.941359,17.958310
2022-01-03 22:46:00,6.682778,210.55800,36.563780,45.743240,164.23856,0.211148,93.841286,6.633695,24.268053,26.066711,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17.997866,17.947008,17.952660
2022-01-03 22:47:00,6.713386,212.94440,37.142994,46.006943,165.82375,0.211242,91.845950,6.590656,24.354885,26.178997,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18.048725,17.839645,17.986565
2022-01-03 22:48:00,6.706837,213.07224,36.940030,46.016360,164.63486,0.214830,89.377530,6.590656,24.437943,26.264862,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18.014818,17.822693,17.947008
2022-01-03 22:49:00,6.670256,214.64897,36.816895,46.167053,161.97398,0.213508,89.707080,6.601983,24.483248,26.304490,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18.020468,17.884853,17.884853


--- 
## 4. Chronological Splitting
To validate properly and avoid time-series data leakage, we split our data chronologically into Train (2022-2024), Validation (H1 2025), and Test (H2 2025 - Jan 2026).

In [5]:
exclude_cols = ['target_5m', 'target_15m', 'target_30m', 'target_60m']
feature_cols = [col for col in df.columns if col not in exclude_cols]
print(f"Number of input features: {len(feature_cols)}")

train_end = pd.to_datetime('2024-12-31 23:59:00')
val_end = pd.to_datetime('2025-06-30 23:59:00')

print(f"Train period: {df.index.min()} to {train_end}")
print(f"Validation period: {train_end + pd.Timedelta(minutes=1)} to {val_end}")
print(f"Test period: {val_end + pd.Timedelta(minutes=1)} to {df.index.max()}")

Number of input features: 132
Train period: 2022-01-03 22:45:00 to 2024-12-31 23:59:00
Validation period: 2025-01-01 00:00:00 to 2025-06-30 23:59:00
Test period: 2025-07-01 00:00:00 to 2026-01-09 23:29:00


--- 
## 5. Model Training & Evaluation (5m, 15m, 30m, 60m horizons)
We train LightGBM models with early stopping. LightGBM excels at time series tabular data.

In [6]:
horizons = [5, 15, 30, 60]
results = {}
test_preds_df = pd.DataFrame(index=df.loc[val_end + pd.Timedelta(minutes=1):].index)
test_preds_df['actual'] = df.loc[val_end + pd.Timedelta(minutes=1):, '03TIC_1023.PV']

for h in horizons:
    print(f"\n{'='*50}")
    print(f" Training Model for Horizon: {h} Minutes")
    print(f"{'='*50}")
    
    target_name = f'target_{h}m'
    
    # Prepare dataset (drop rows with NaN targets)
    data_h = df[feature_cols + [target_name]].dropna(subset=[target_name])
    
    # Chronological splits
    train_data = data_h.loc[:train_end]
    val_data = data_h.loc[train_end + pd.Timedelta(minutes=1):val_end]
    test_data = data_h.loc[val_end + pd.Timedelta(minutes=1):]
    
    X_train, y_train = train_data[feature_cols], train_data[target_name]
    X_val, y_val = val_data[feature_cols], val_data[target_name]
    X_test, y_test = test_data[feature_cols], test_data[target_name]
    
    print(f"Train shape: {X_train.shape}, Val shape: {X_val.shape}, Test shape: {X_test.shape}")
    
    # LightGBM Regressor setup
    model = lgb.LGBMRegressor(
        n_estimators=1000,
        learning_rate=0.05,
        num_leaves=63,
        max_depth=8,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )
    
    start_time = time.time()
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric='rmse',
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )
    print(f"Training finished in {time.time() - start_time:.2f} seconds.")
    
    # Save model
    model_path = os.path.join(MODEL_DIR, f"lgb_model_{h}m.pkl")
    joblib.dump(model, model_path)
    
    # Predict
    val_preds = model.predict(X_val)
    test_preds = model.predict(X_test)
    
    test_preds_df[f'pred_{h}m'] = pd.Series(test_preds, index=X_test.index)
    
    # Regression metrics
    val_mae = mean_absolute_error(y_val, val_preds)
    val_rmse = np.sqrt(mean_squared_error(y_val, val_preds))
    test_mae = mean_absolute_error(y_test, test_preds)
    test_rmse = np.sqrt(mean_squared_error(y_test, test_preds))
    
    # Classification metrics (Threshold = 21.0)
    threshold = 21.0
    y_test_alarm = (y_test >= threshold).astype(int)
    test_preds_alarm = (test_preds >= threshold).astype(int)
    
    precision, recall, f1, _ = precision_recall_fscore_support(y_test_alarm, test_preds_alarm, average='binary', zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_test_alarm, test_preds_alarm).ravel()
    far = fp / (fp + tn) if (fp + tn) > 0 else 0
    
    print(f"\n--- Regression Metrics ({h}m) ---")
    print(f"Val MAE: {val_mae:.4f} | Val RMSE: {val_rmse:.4f}")
    print(f"Test MAE: {test_mae:.4f} | Test RMSE: {test_rmse:.4f}")
    
    print(f"\n--- Alarm Classification Metrics (Threshold={threshold}) ---")
    print(f"Precision (Accuracy of alarms): {precision:.4%}")
    print(f"Recall (Catch rate / Sensitivity): {recall:.4%}")
    print(f"F1-Score (Balanced): {f1:.4%}")
    print(f"False Alarm Rate (FAR): {far:.4%}")
    print(f"Confusion Matrix: TP={tp}, FP={fp}, TN={tn}, FN={fn}")
    
    results[h] = {
        'val_mae': val_mae, 'val_rmse': val_rmse,
        'test_mae': test_mae, 'test_rmse': test_rmse,
        'precision': precision, 'recall': recall, 'f1': f1, 'far': far
    }

# Save predictions
test_preds_df.to_parquet(os.path.join(OUTPUT_DIR, "test_predictions.parquet"))
print("\nAll models trained and predictions saved.")


 Training Model for Horizon: 15 Minutes
Train shape: (1495205, 132), Val shape: (252923, 132), Test shape: (271371, 132)
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.611729 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 32939
[LightGBM] [Info] Number of data points in the train set: 1495205, number of used features: 132
[LightGBM] [Info] Start training from score 17.839589
Training finished in 76.85 seconds.

--- Regression Metrics (15m) ---
Val MAE: 0.1414 | Val RMSE: 0.2454
Test MAE: 0.2283 | Test RMSE: 0.4052

--- Alarm Classification Metrics (Threshold=21.0) ---
Precision (Accuracy of alarms): 93.3673%
Recall (Catch rate / Sensitivity): 90.3976%
F1-Score (Balanced): 91.8585%
False Alarm Rate (FAR): 0.2026%
Confusion Matrix: TP=7503, FP=533, TN=262538, FN=797

 Training Model for Horizon: 30 Minutes
Train shape: (1495205, 132), Val shape: (252923, 132), Test shape: (271356, 132)
[LightGBM] [

In [12]:
list(train_data.columns)

['02FI_1000.PV',
 '03FIC_1085.PV',
 '03LIC_1016.PV',
 '03LIC_1034.PV',
 '03PDI_1020.PV',
 '03PDI_1077.PV',
 '03PIC_1013.PV',
 '03PIC_1023.PV',
 '03PIC_1068.PV',
 '03PI_1409.PV',
 '03TIC_1009.PV',
 '03TIC_1023.PV',
 '03TIC_1145.PV',
 '03TI_1002.PV',
 '03TI_1005.PV',
 '03TI_1015.PV',
 '03TI_1024.PV',
 '03TI_1081.PV',
 '03TI_1108.PV',
 'hour',
 'month',
 'dayofweek',
 '03TIC_1023.PV_lag_1',
 '03TIC_1023.PV_lag_2',
 '03TIC_1023.PV_lag_5',
 '03TIC_1023.PV_lag_10',
 '03TIC_1023.PV_lag_15',
 '03TIC_1023.PV_lag_30',
 '03TIC_1023.PV_lag_60',
 '03TI_1024.PV_lag_1',
 '03TI_1024.PV_lag_2',
 '03TI_1024.PV_lag_5',
 '03TI_1024.PV_lag_10',
 '03TI_1024.PV_lag_15',
 '03TI_1024.PV_lag_30',
 '03TI_1024.PV_lag_60',
 '03TI_1015.PV_lag_1',
 '03TI_1015.PV_lag_2',
 '03TI_1015.PV_lag_5',
 '03TI_1015.PV_lag_10',
 '03TI_1015.PV_lag_15',
 '03TI_1015.PV_lag_30',
 '03TI_1015.PV_lag_60',
 '03PIC_1023.PV_lag_1',
 '03PIC_1023.PV_lag_2',
 '03PIC_1023.PV_lag_5',
 '03PIC_1023.PV_lag_10',
 '03PIC_1023.PV_lag_15',
 '03PIC_1

--- 
## 6. Summary of Results

In [7]:
summary_df = pd.DataFrame(results).T
print(summary_df)

     val_mae  val_rmse  test_mae  test_rmse  precision    recall        f1  \
15  0.141441  0.245420  0.228258   0.405243   0.933673  0.903976  0.918585   
30  0.195171  0.343696  0.317782   0.534371   0.901770  0.871566  0.886411   
60  0.298327  0.483614  0.441207   0.680712   0.841201  0.790120  0.814861   

         far  
15  0.002026  
30  0.002996  
60  0.004707  
